# Encoder–Decoder Image Captioning with ViT and BLIP

This notebook demonstrates visual feature extraction with Vision Transformer (ViT) and image caption generation with the pretrained BLIP encoder–decoder model. Run the cells in order.

In [ ]:
import torch
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import transforms

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    BlipProcessor,
    BlipForConditionalGeneration
)

In [ ]:
from pathlib import Path

image_candidates = [Path("cat.jpeg"), Path("../cat.jpeg")]
image_path = next((path for path in image_candidates if path.exists()), None)

if image_path is None:
    raise FileNotFoundError("cat.jpeg was not found. Keep the sample image in the repository root.")

image = Image.open(image_path).convert("RGB")

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title("Original Image")
plt.show()

In [ ]:
processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224"
)

model.eval()

In [ ]:
inputs = processor(
    images=image,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model.vit(**inputs)

In [ ]:
outputs.last_hidden_state.shape

In [ ]:
embedding = outputs.last_hidden_state[:, 0, :]

print("Embedding shape:", embedding.shape)

In [ ]:
import numpy as np

img = np.array(image.resize((224, 224)))

plt.figure(figsize=(7, 7))
plt.imshow(img)

for x in range(0, 225, 16):
    plt.axvline(x, linewidth=0.5)

for y in range(0, 225, 16):
    plt.axhline(y, linewidth=0.5)

plt.title("Image Divided into ViT Patches")
plt.axis("off")
plt.show()

In [ ]:
processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model.eval()

In [ ]:
inputs = processor(
    images=image,
    return_tensors="pt"
)

In [ ]:
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=30
    )

In [ ]:
caption = processor.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("Generated Caption:")
print(caption)

In [ ]:
plt.figure(figsize=(8, 6))

plt.imshow(image)
plt.axis("off")

plt.title(
    f"Encoder-Decoder Output\n\n{caption}",
    fontsize=14
)

plt.show()